In [3]:
"""
用户研究数据分析 - 停留时间统计
支持SQL和CSV文件格式
"""

import pandas as pd
import numpy as np
import re

def load_data_from_sql(file_path):
    """
    从SQL文件中解析INSERT语句并加载数据
    """
    print(f"正在从SQL文件加载数据: {file_path}")
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # 检查是否包含INSERT语句
        if 'INSERT' not in content.upper():
            raise Exception("SQL文件中没有找到INSERT语句")
        
        print("找到INSERT语句，开始解析...")
        data = []
        
        # 查找所有包含logs表的INSERT语句
        # 匹配模式: INSERT INTO `logs` VALUES (...)
        insert_pattern = r"INSERT INTO.*?logs.*?VALUES\s*(.*?);"
        matches = re.findall(insert_pattern, content, re.IGNORECASE | re.DOTALL)
        
        print(f"找到 {len(matches)} 条INSERT语句")
        
        for match in matches:
            # 提取所有的行数据 (value1, value2, ...), (value1, value2, ...)
            # 使用正则表达式匹配括号内的内容
            rows_pattern = r'\((.*?)\)(?=,\s*\(|\s*$)'
            rows = re.findall(rows_pattern, match, re.DOTALL)
            
            for row in rows:
                # 分割每行的值
                # 处理可能包含逗号的引号内容
                values = []
                current_value = ""
                in_quotes = False
                quote_char = None
                
                for char in row + ',':
                    if char in ('"', "'") and (not in_quotes or char == quote_char):
                        if not in_quotes:
                            in_quotes = True
                            quote_char = char
                        else:
                            in_quotes = False
                            quote_char = None
                    elif char == ',' and not in_quotes:
                        values.append(current_value.strip().strip('"').strip("'"))
                        current_value = ""
                    else:
                        current_value += char
                
                # 清理最后一个值
                if current_value.strip():
                    values.append(current_value.strip().strip('"').strip("'"))
                
                # 只保留有10个字段的行（假设是正确的数据行）
                if len(values) == 10:
                    data.append(values)
        
        if not data:
            raise Exception("无法从SQL文件中提取数据")
        
        print(f"成功提取 {len(data)} 行数据")
        
        # 创建DataFrame
        df = pd.DataFrame(data, columns=['id', 'user_id', 'qid', 'docno', 'event_type', 
                                        'start_idx', 'end_idx', 'duration', 'pass_flag', 'timestamp'])
        
        # 转换数据类型
        df['id'] = pd.to_numeric(df['id'], errors='coerce')
        df['qid'] = pd.to_numeric(df['qid'], errors='coerce')
        df['start_idx'] = pd.to_numeric(df['start_idx'], errors='coerce')
        df['end_idx'] = pd.to_numeric(df['end_idx'], errors='coerce')
        df['duration'] = pd.to_numeric(df['duration'], errors='coerce')
        df['pass_flag'] = pd.to_numeric(df['pass_flag'], errors='coerce')
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
        
        print(f"数据加载完成！DataFrame形状: {df.shape}")
        print(f"\n数据预览:")
        print(df.head())
        
        return df
        
    except Exception as e:
        print(f"加载SQL文件时出错: {e}")
        return None


def load_data(file_path):
    """
    智能加载数据：自动识别SQL或CSV文件
    """
    if file_path.endswith('.sql'):
        return load_data_from_sql(file_path)
    elif file_path.endswith('.csv'):
        print(f"正在从CSV文件加载数据: {file_path}")
        try:
            df = pd.read_csv(file_path)
            print(f"数据加载完成！DataFrame形状: {df.shape}")
            return df
        except Exception as e:
            print(f"加载CSV文件时出错: {e}")
            return None
    else:
        print("不支持的文件格式。请使用.sql或.csv文件")
        return None


def analyze_event_duration(df):
    """
    分析PASSAGE_SELECTION和OPEN_DOC事件的停留时间统计
    """
    if df is None:
        print("错误：数据框为空")
        return None, None, None
    
    # 确保event_type是字符串类型
    df['event_type'] = df['event_type'].astype(str)
    
    # 确保duration是数值类型
    df['duration'] = pd.to_numeric(df['duration'], errors='coerce')
    
    # 筛选PASSAGE_SELECTION事件
    passage_selection = df[df['event_type'] == 'PASSAGE_SELECTION'].copy()
    
    # 筛选OPEN_DOC事件
    open_doc = df[df['event_type'] == 'OPEN_DOC'].copy()
    
    # 计算统计数据
    results = {}
    
    print("\n" + "="*80)
    print("用户研究 - 事件停留时间分析")
    print("="*80)
    
    # PASSAGE_SELECTION统计
    if len(passage_selection) > 0:
        ps_duration_ms = passage_selection['duration']
        ps_duration_sec = ps_duration_ms / 1000
        
        results['passage_selection'] = {
            '事件数量': len(passage_selection),
            '平均时长(秒)': ps_duration_sec.mean(),
            '中位数(秒)': ps_duration_sec.median(),
            '平均时长(毫秒)': ps_duration_ms.mean(),
            '中位数(毫秒)': ps_duration_ms.median(),
            '标准差(秒)': ps_duration_sec.std(),
            '最小值(秒)': ps_duration_sec.min(),
            '最大值(秒)': ps_duration_sec.max(),
            '25分位数(秒)': ps_duration_sec.quantile(0.25),
            '75分位数(秒)': ps_duration_sec.quantile(0.75)
        }
        
        print("\n📊 PASSAGE_SELECTION 事件分析:")
        print(f"   事件总数:     {len(passage_selection):,}")
        print(f"   平均停留时间: {ps_duration_sec.mean():.2f} 秒 ({ps_duration_ms.mean():.2f} 毫秒)")
        print(f"   中位停留时间: {ps_duration_sec.median():.2f} 秒 ({ps_duration_ms.median():.2f} 毫秒)")
        print(f"   标准差:       {ps_duration_sec.std():.2f} 秒")
        print(f"   最小值:       {ps_duration_sec.min():.2f} 秒")
        print(f"   最大值:       {ps_duration_sec.max():.2f} 秒")
        print(f"   25分位数:     {ps_duration_sec.quantile(0.25):.2f} 秒")
        print(f"   75分位数:     {ps_duration_sec.quantile(0.75):.2f} 秒")
    else:
        print("\n⚠️  警告: 没有找到PASSAGE_SELECTION事件")
        results['passage_selection'] = None
    
    # OPEN_DOC统计
    if len(open_doc) > 0:
        od_duration_ms = open_doc['duration']
        od_duration_sec = od_duration_ms / 1000
        
        results['open_doc'] = {
            '事件数量': len(open_doc),
            '平均时长(秒)': od_duration_sec.mean(),
            '中位数(秒)': od_duration_sec.median(),
            '平均时长(毫秒)': od_duration_ms.mean(),
            '中位数(毫秒)': od_duration_ms.median(),
            '标准差(秒)': od_duration_sec.std(),
            '最小值(秒)': od_duration_sec.min(),
            '最大值(秒)': od_duration_sec.max(),
            '25分位数(秒)': od_duration_sec.quantile(0.25),
            '75分位数(秒)': od_duration_sec.quantile(0.75)
        }
        
        print("\n📄 OPEN_DOC 事件分析:")
        print(f"   事件总数:     {len(open_doc):,}")
        print(f"   平均停留时间: {od_duration_sec.mean():.2f} 秒 ({od_duration_ms.mean():.2f} 毫秒)")
        print(f"   中位停留时间: {od_duration_sec.median():.2f} 秒 ({od_duration_ms.median():.2f} 毫秒)")
        print(f"   标准差:       {od_duration_sec.std():.2f} 秒")
        print(f"   最小值:       {od_duration_sec.min():.2f} 秒")
        print(f"   最大值:       {od_duration_sec.max():.2f} 秒")
        print(f"   25分位数:     {od_duration_sec.quantile(0.25):.2f} 秒")
        print(f"   75分位数:     {od_duration_sec.quantile(0.75):.2f} 秒")
    else:
        print("\n⚠️  警告: 没有找到OPEN_DOC事件")
        results['open_doc'] = None
    
    print("\n" + "="*80 + "\n")
    
    return results, passage_selection, open_doc


def export_results_to_csv(results, filename='duration_summary.csv'):
    """
    将结果导出为CSV文件
    """
    data = []
    for event_type, stats in results.items():
        if stats is not None:
            row = {'事件类型': event_type}
            row.update(stats)
            data.append(row)
    
    if data:
        results_df = pd.DataFrame(data)
        results_df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"✓ 统计结果已保存到 '{filename}'")
        return results_df
    return None


# 主程序
if __name__ == "__main__":
    # 使用你的SQL文件
    df = load_data('backup_qe_v2_full.sql')
    
    if df is not None:
        # 运行分析
        results, ps_df, od_df = analyze_event_duration(df)
    else:
        print("\n数据加载失败，请检查文件路径和格式。")

正在从SQL文件加载数据: backup_qe_v2_full.sql
找到INSERT语句，开始解析...
找到 6 条INSERT语句
成功提取 48881 行数据
数据加载完成！DataFrame形状: (48881, 10)

数据预览:
   id                   user_id   qid                         docno  \
0  74  6357a92c14a05d21ebdd897f  2082                             0   
1  75  627286792f25495f123cc27b  2082                             0   
2  76  5c9d88bb2b3c77001744ec1c  2082                             0   
3  77  6357a92c14a05d21ebdd897f  2082  msmarco_passage_39_461879190   
4  78  666a1e606b80f9865c1f3d5f  2082                             0   

      event_type  start_idx  end_idx  duration  pass_flag           timestamp  
0  BOT_DETECTION         -1       -1      7580          1 2025-04-21 12:26:53  
1  BOT_DETECTION         -1       -1     12339          1 2025-04-21 12:27:41  
2  BOT_DETECTION         -1       -1      9491          1 2025-04-21 12:27:45  
3       OPEN_DOC         -1       -1     55110          0 2025-04-21 12:27:48  
4  BOT_DETECTION         -1       -1      6717   